# SH01 complete local AI video-editor acceptance test

This compact notebook verifies that the server edits video beyond
captions and silence cuts:

1. one automatic AI-directed editorial run;
2. one deterministic compositor run that must render graphics,
   camera motion, and transitions;
3. local supplementary-media/audio branches when available;
4. receipt, quality, motion, storyboard, and archive checks.

It fails when a feature exists only in the plan but was not
rendered into the final MP4.

## 1. Clone and install before importing Torch

In [ ]:
%%bash
set -e
REPO_DIR=/kaggle/working/sh01-fastapi-server
REPO_REF="${AI_COMPOSE_REPO_REF:-feature/ai-compose-kaggle}"
REPO_URL="${AI_COMPOSE_REPO_URL:-https://github.com/BarrieBretton/sh01-fastapi-server.git}"

if [ -d "$REPO_DIR/.git" ]; then
    if [ -n "$(git -C "$REPO_DIR" status --porcelain)" ]; then
        echo "Retaining dirty checkout: $REPO_DIR"
    else
        git -C "$REPO_DIR" fetch origin "$REPO_REF"
        git -C "$REPO_DIR" checkout "$REPO_REF"
        git -C "$REPO_DIR" pull --ff-only origin "$REPO_REF"
    fi
else
    git clone -b "$REPO_REF" "$REPO_URL" "$REPO_DIR"
fi

cd "$REPO_DIR"
if [ -f requirements.txt ]; then python -m pip install -q -r requirements.txt; fi
python -m pip install -q -r requirements-ai-compose.txt

## 2. GPU/package check

In [ ]:
import torch
import bitsandbytes

print("Torch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
assert torch.cuda.is_available(), "Enable the Kaggle GPU"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

## 3. Load private environment and source

In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import py_compile
import shutil
import subprocess
import sys
import time
import uuid
import zipfile
from pathlib import Path

import requests
from dotenv import load_dotenv

REPO = Path("/kaggle/working/sh01-fastapi-server").resolve()
ENV_PATH = REPO / ".env"
if not ENV_PATH.is_file():
    raise FileNotFoundError(f"Create/load the private .env first: {ENV_PATH}")
load_dotenv(ENV_PATH, override=True)

patch_zip = os.environ.get("AI_COMPOSE_PATCH_ZIP", "").strip()
if patch_zip:
    with zipfile.ZipFile(Path(patch_zip).expanduser().resolve()) as archive:
        for member in archive.infolist():
            target = (REPO / member.filename).resolve()
            if target != REPO and REPO not in target.parents:
                raise RuntimeError(f"Unsafe patch member: {member.filename}")
            archive.extract(member, REPO)
    print("Applied root patch:", patch_zip)

os.environ.setdefault("MEDIA_JOB_STORE_MODE", "local")
os.environ.setdefault("MEDIA_QUEUE_CONSUMER_ENABLED", "false")
os.environ.setdefault("AI_COMPOSE_HF_MODEL", "Qwen/Qwen2.5-3B-Instruct")
os.environ.setdefault("AI_COMPOSE_HF_LOAD_4BIT", "true")
os.environ.setdefault("AI_COMPOSE_HF_MAX_NEW_TOKENS", "1800")
os.environ.setdefault("AI_COMPOSE_ALLOW_DETERMINISTIC_RECOVERY", "true")
os.environ.setdefault("AI_COMPOSE_WORK_DIR", "/kaggle/working/ai-compose-jobs")
os.environ.setdefault("MEDIA_ALLOW_UNLIMITED_RESOURCES", "true")

source_value = os.environ.get("AI_COMPOSE_SOURCE_PATH", "").strip()
if not source_value:
    raise RuntimeError("Set AI_COMPOSE_SOURCE_PATH in the private .env")
SOURCE = Path(source_value).expanduser().resolve()
if not SOURCE.is_file():
    raise FileNotFoundError(SOURCE)
os.environ["MEDIA_LOCAL_SOURCE_DIR"] = str(SOURCE.parent)

API_KEY = os.environ.get("MEDIA_CLIP_API_KEY", "").strip()
if not API_KEY:
    raise RuntimeError("MEDIA_CLIP_API_KEY is missing from the private .env")

BASE = "http://127.0.0.1:5000"
HEADERS = {"X-API-KEY": API_KEY}
WORK_ROOT = Path(os.environ["AI_COMPOSE_WORK_DIR"])
OUTPUT_ROOT = Path("/kaggle/working/sh01-complete-editor")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

required = [
    "ai_compose.py", "ai_compose_engine.py", "ai_compose_schema.py",
    "ai_compose_policy.py", "ai_compose_ir.py",
    "ai_compose_quality.py", "ai_compose_refinement.py",
]
for name in required:
    path = REPO / name
    if not path.is_file():
        raise FileNotFoundError(path)
    py_compile.compile(str(path), doraise=True)

sys.path.insert(0, str(REPO))
from ai_compose_schema import CompositionPlan
assert "graphics" in CompositionPlan.model_fields
assert importlib.util.find_spec("json_repair")

SERVER_LOG = Path("/kaggle/working/sh01_complete_editor_server.log")
SERVER_LOG_HANDLE = SERVER_LOG.open("w", encoding="utf-8")
SERVER_PROCESS = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app:app",
     "--host", "0.0.0.0", "--port", "5000", "--workers", "1"],
    cwd=str(REPO), env=os.environ.copy(),
    stdout=SERVER_LOG_HANDLE, stderr=subprocess.STDOUT, text=True,
)
deadline = time.monotonic() + 180
while time.monotonic() < deadline:
    if SERVER_PROCESS.poll() is not None:
        raise RuntimeError(SERVER_LOG.read_text(encoding="utf-8", errors="replace")[-8000:])
    try:
        if requests.get(f"{BASE}/openapi.json", timeout=5).ok:
            print("Server ready; PID:", SERVER_PROCESS.pid)
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    raise TimeoutError("Server did not become ready")

In [ ]:
from IPython.display import Audio, display
import numpy as np

# Audible checkpoint: server preflight is complete
sample_rate = 8000
t = np.linspace(0, 0.35, int(sample_rate * 0.35), endpoint=False)
display(Audio(np.sin(2 * np.pi * 440 * t), rate=sample_rate, autoplay=True))

## 4. Capability contract and reusable job helpers

In [ ]:
capabilities = requests.get(
    f"{BASE}/media/ai-compose-capabilities",
    headers=HEADERS, timeout=30,
).json()
required_features = {
    "versioned-edit-plan",
    "rendered-motion-graphics",
    "animated-camera-keyframes",
    "xfade-transitions",
    "technical-quality-review",
}
missing = required_features.difference(capabilities.get("features", []))
if missing:
    raise RuntimeError(f"Rendered editor patch missing: {sorted(missing)}")
print(json.dumps(capabilities, indent=2))

JOB_RECEIPTS = []

def submit(payload):
    payload = dict(payload)
    payload.setdefault("request_id", f"complete-editor-{uuid.uuid4()}")
    response = requests.post(
        f"{BASE}/media/ai-compose",
        headers={**HEADERS, "Content-Type": "application/json"},
        json=payload, timeout=120,
    )
    print("submit:", response.status_code)
    if not response.ok:
        print(response.text[:4000])
    response.raise_for_status()
    return response.json()

def poll(job, timeout_seconds=3600):
    deadline = time.monotonic() + timeout_seconds
    previous = None
    while time.monotonic() < deadline:
        status = requests.get(
            f"{BASE}/media/ai-compose-jobs/{job['job_id']}",
            headers=HEADERS, timeout=60,
        ).json()
        marker = (
            status.get("status"), status.get("stage"),
            status.get("progress"), status.get("refinement_round"),
            status.get("error"),
        )
        if marker != previous:
            print(
                "status=", status.get("status"),
                "stage=", status.get("stage"),
                "progress=", status.get("progress"),
                "round=", status.get("refinement_round"),
                "error=", status.get("error"),
            )
            previous = marker
        if status.get("status") in {"complete", "failed"}:
            JOB_RECEIPTS.append(status)
            if status["status"] != "complete":
                raise RuntimeError(json.dumps(status, indent=2))
            return status
        time.sleep(5)
    raise TimeoutError(job["job_id"])

def download(status, variant=1):
    destination = Path("/kaggle/working") / (
        f"complete-editor-{status['job_id']}-{variant:02d}.mp4"
    )
    response = requests.get(
        f"{BASE}/media/ai-compose-files/{status['job_id']}/"
        f"variant_{variant:02d}.mp4",
        headers=HEADERS, timeout=900,
    )
    response.raise_for_status()
    destination.write_bytes(response.content)
    return destination

def load_receipt(status):
    return json.loads(
        (WORK_ROOT / status["job_id"] / "composition_receipt.json")
        .read_text(encoding="utf-8")
    )

def show_video(path):
    from IPython.display import Video, display
    display(Video(str(path), embed=True, width=420))

## 5. Test A — one-prompt automatic editorial decision

In [ ]:
local_library = os.environ.get("AI_COMPOSE_LOCAL_ASSET_LIBRARY", "").strip()
has_remote_assets = any(
    os.environ.get(key, "").strip()
    for key in ("PEXELS_API_KEY", "PIXABAY_API_KEY", "UNSPLASH_ACCESS_KEY")
)
auto_payload = {
    "unlimited_resources": True,
    "source_path": str(SOURCE),
    "brief": (
        "Edit this into the best possible short-form video. Decide from "
        "the source whether it needs hook graphics, motivated camera "
        "movement, supporting visuals, SFX, music, or voiceover. "
        "Preserve the original speaker when dialogue is the narrative. "
        "Do not add competing audio."
    ),
    "preset": "viral_short",
    "target_seconds": 15,
    "output_count": 1,
    "captions_enabled": True,
    "fetch_assets": bool(local_library or has_remote_assets),
    "local_asset_library": local_library or None,
    "max_asset_layers": 2,
    "llm_required": True,
    "refinement_iterations": 1,
    "editorial_policy": {
        "content_type": "auto",
        "editing_intensity": "auto",
        "cut_policy": "auto",
        "source_audio": "auto",
        "voiceover": "auto",
        "supplementary_visuals": "auto",
        "motion_graphics": "auto",
        "camera_motion": "auto",
        "music": "auto",
        "sfx": "auto",
        "review_level": "strict",
    },
    "sfx": {"mode": "adaptive"},
}
auto_status = poll(submit(auto_payload), timeout_seconds=3600)
auto_video = download(auto_status)
auto_receipt = load_receipt(auto_status)
print(json.dumps({
    "editorial_profile": auto_receipt.get("editorial_profile"),
    "plan": auto_receipt.get("plan"),
    "resolved_layers": len(auto_receipt.get("resolved_layers", [])),
}, indent=2))
show_video(auto_video)

## 6. Test B — rendered compositor acceptance

In [ ]:
compositor_payload = {
    "unlimited_resources": True,
    "source_path": str(SOURCE),
    "brief": (
        "Create a polished vertical short with a strong opening hook, "
        "a concise explanatory bridge callout, motivated punch-in "
        "camera movement, and a tasteful transition between selected "
        "moments. Preserve dialogue and captions. Never add voiceover, "
        "music, SFX, or external assets."
    ),
    "preset": "documentary",
    "target_seconds": 15,
    "output_count": 1,
    "captions_enabled": True,
    "fetch_assets": False,
    "llm_required": False,
    "refinement_iterations": 0,
    "editorial_policy": {
        "content_type": "documentary",
        "editing_intensity": "cinematic",
        "source_audio": "preserve",
        "voiceover": "off",
        "supplementary_visuals": "off",
        "motion_graphics": "hero",
        "camera_motion": "dynamic",
        "music": "off",
        "sfx": "off",
        "review_level": "strict",
    },
    "sfx": {"mode": "off"},
}
compositor_status = poll(submit(compositor_payload), timeout_seconds=2400)
compositor_video = download(compositor_status)
compositor_receipt = load_receipt(compositor_status)
compositor_review = compositor_receipt["quality_review"]["variants"]["1"]["render"]
feature_check = compositor_review["checks"]["render_features"]
print(json.dumps({
    "actual": feature_check["actual"],
    "required": feature_check["required"],
    "quality_ok": compositor_review["ok"],
}, indent=2))
assert compositor_review["ok"], compositor_review
for feature, minimum in feature_check["required"].items():
    assert feature_check["actual"].get(feature, 0) >= minimum, feature_check
show_video(compositor_video)

In [ ]:
from IPython.display import Audio, display
import numpy as np

# Audible checkpoint: automatic editing and compositor acceptance tests are complete
sample_rate = 8000
t = np.linspace(0, 0.35, int(sample_rate * 0.35), endpoint=False)
display(Audio(np.sin(2 * np.pi * 440 * t), rate=sample_rate, autoplay=True))

## 7. Test C — local supplementary media and available SFX

In [ ]:
optional_results = {}
if local_library:
    asset_payload = dict(compositor_payload)
    asset_payload.update({
        "brief": (
            "Add one or two highly relevant supplementary visuals as "
            "partial-frame overlays. Keep the speaker face and captions visible."
        ),
        "fetch_assets": True,
        "local_asset_library": local_library,
        "asset_sources": {
            "local_library": True, "pexels": False,
            "pixabay": False, "unsplash": False, "freesound": False,
        },
        "max_asset_layers": 2,
        "editorial_policy": {
            **compositor_payload["editorial_policy"],
            "supplementary_visuals": "require",
            "motion_graphics": "selective",
        },
    })
    asset_status = poll(submit(asset_payload), timeout_seconds=2400)
    asset_receipt = load_receipt(asset_status)
    optional_results["resolved_layers"] = len(asset_receipt.get("resolved_layers", []))
    assert optional_results["resolved_layers"] > 0, asset_receipt
    show_video(download(asset_status))
else:
    print("Local asset branch skipped: AI_COMPOSE_LOCAL_ASSET_LIBRARY is unset")

sfx_library = os.environ.get("AI_COMPOSE_LOCAL_SFX_LIBRARY", "").strip()
if sfx_library and Path(sfx_library).exists():
    audio_payload = dict(compositor_payload)
    audio_payload.update({
        "editorial_policy": {
            **compositor_payload["editorial_policy"],
            "sfx": "transitions_only",
        },
        "sfx": {
            "mode": "transitions_only",
            "local_library": sfx_library,
            "volume": 0.4,
        },
        "local_sfx_library": sfx_library,
    })
    audio_status = poll(submit(audio_payload), timeout_seconds=2400)
    optional_results["sfx_job"] = audio_status["job_id"]
    show_video(download(audio_status))
else:
    print("SFX branch skipped: AI_COMPOSE_LOCAL_SFX_LIBRARY is unset")
print("Optional results:", optional_results)

## 8. Actual-output QA, storyboard, archive, and shutdown

In [ ]:
from IPython.display import display
from PIL import Image, ImageDraw, ImageOps
import numpy as np

def storyboard(video_path, label, count=6):
    probe = json.loads(subprocess.run(
        ["ffprobe", "-v", "error", "-show_format", "-of", "json", str(video_path)],
        capture_output=True, text=True, check=True,
    ).stdout)
    duration = float(probe["format"]["duration"])
    root = OUTPUT_ROOT / "storyboards" / label
    root.mkdir(parents=True, exist_ok=True)
    thumbs = []
    for index in range(count):
        timestamp = duration * (index + 0.5) / count
        frame = root / f"frame_{index:02d}.jpg"
        subprocess.run([
            "ffmpeg", "-y", "-loglevel", "error", "-ss", str(timestamp),
            "-i", str(video_path), "-frames:v", "1",
            "-vf", "scale=240:-1", str(frame),
        ], check=True)
        image = Image.open(frame).convert("RGB")
        canvas = Image.new("RGB", (260, 300), "#111111")
        canvas.paste(ImageOps.contain(image, (240, 260)), (10, 10))
        ImageDraw.Draw(canvas).text((10, 278), f"t={timestamp:.2f}s", fill="white")
        thumbs.append(canvas)
    sheet = Image.new("RGB", (780, 600), "black")
    for index, thumb in enumerate(thumbs):
        sheet.paste(thumb, ((index % 3) * 260, (index // 3) * 300))
    sheet.save(root / "contact_sheet.jpg")
    display(sheet)

storyboard(compositor_video, "compositor")
print("Automatic job:", auto_status["job_id"])
print("Compositor job:", compositor_status["job_id"])
print("Output root:", OUTPUT_ROOT)

for status in JOB_RECEIPTS:
    job_root = WORK_ROOT / status["job_id"]
    destination = OUTPUT_ROOT / status["job_id"]
    destination.mkdir(parents=True, exist_ok=True)
    for output in status.get("outputs", []):
        source = Path(output.get("local_path", ""))
        if source.is_file():
            shutil.copy2(source, destination / source.name)
    for name in ("composition_receipt.json", "evidence.json"):
        source = job_root / name
        if source.is_file():
            shutil.copy2(source, destination / name)

archive = shutil.make_archive(
    "/kaggle/working/sh01-complete-editor-results", "zip", OUTPUT_ROOT
)
print("Download archive:", archive)

if SERVER_PROCESS.poll() is None:
    SERVER_PROCESS.terminate()
    SERVER_PROCESS.wait(timeout=20)
SERVER_LOG_HANDLE.close()
print("Server stopped")

In [ ]:
from IPython.display import Audio, display
import numpy as np

# Audible checkpoint: complete editor acceptance suite is finished
sample_rate = 8000
t = np.linspace(0, 0.35, int(sample_rate * 0.35), endpoint=False)
display(Audio(np.sin(2 * np.pi * 440 * t), rate=sample_rate, autoplay=True))